In [1]:
import openmm as mm
from openmm import app, unit
from openmm.app.gromacsgrofile import GromacsGroFile

import martini_openmm as martini

import numpy as np
import sys

from openmmnapshift.utils import RESIDUE_TYPES, get_napshift_force

## Define simulation parameters

In [ ]:
temperature = 298*unit.kelvin
timestep = 10*unit.femtosecond
pressure = 1*unit.bar

max_K = 25
K_gradient = 0.001

report_interval = 1000

## Set up Martini3 system

In [ ]:
conf = GromacsGroFile(f"Data/2ND3/system.gro")
box_vectors = conf.getPeriodicBoxVectors()
topfile = martini.MartiniTopFile(
    f"Data/2ND3/system.top",
    periodicBoxVectors=box_vectors,
    epsilon_r=15,
)
topology = topfile.topology
system = topfile.create_system(nonbonded_cutoff=1.1 * unit.nanometer)
barostat = mm.MonteCarloBarostat(pressure, temperature)
system.addForce(barostat)

## Add Chemical Shift restraints

In [ ]:
napshift_force = get_napshift_force(topology, 'Data/2ND3/CS.txt', model_type='martini')
napshift_force.setUsesPeriodicBoundaryConditions(True)
system.addForce(napshift_force)

## Create the OpenMM simulation

In [ ]:
integrator = mm.LangevinMiddleIntegrator(temperature, 0.01/unit.picosecond, timestep)
platform = mm.Platform.getPlatformByName('CUDA')
simulation = app.Simulation(topology, system, integrator, platform, {"Precision" : "mixed", 'DeviceIndex' : "0"})
simulation.context.setPositions(conf.getPositions())
simulation.minimizeEnergy()
simulation.context.setVelocitiesToTemperature(temperature)


## Add reporters

In [ ]:
xtc_reporter = app.XTCReporter('Data/2ND3/output.xtc', report_interval, append=False, enforcePeriodicBox=True, atomSubset=[atom.index for atom in topology.atoms() if atom.residue.name in RESIDUE_TYPES.keys()])
state_data_reporter = app.StateDataReporter(sys.stdout, report_interval, step=True, time=True, potentialEnergy=True, kineticEnergy=True, totalEnergy=True, temperature=True, volume=True, speed=True)
simulation.reporters.append(xtc_reporter)
simulation.reporters.append(state_data_reporter)

## Simulate!

In [3]:
print(f"Simulating without CS restraints")
simulation.step(100000)

warmup_steps = int(np.floor(max_K/K_gradient))
print(f"Warming up CS restraints for {len(range(warmup_steps))} steps")
for i in range(warmup_steps):
    simulation.step(1)
    simulation.context.setParameter('NapShift_K', (i*K_gradient))
    
print(f"Simulating with CS restraints")
simulation.step(100000)

/home/mina/anaconda3/envs/OpenMMNapShift/lib/python3.11/site-packages/pycamcoil/camcoil_engine.py:108: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/anaconda3/envs/OpenMMNapShift/lib/python3.11/site-packages/pycamcoil/camcoil_engine.py:108: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/anaconda3/envs/OpenMMNapShift/lib/python3.11/site-packages/pycamcoil/camcoil_engine.py:108: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mina/anaconda3/envs/OpenMMNapShift/lib/pyth

Successfully added NapShift Force
